# 448 px ViT + attribute model, tuned on validation (RQ3)

[`consolidation.ipynb`](consolidation.ipynb) chose the attribute partner and weight `w` with the **224 px ViT**, because only that network had saved **validation** probabilities. The 448 px ViT is the better network (validation macro-F1 0.814 vs 0.789), but its partner and weight were only carried over from the 224 px choice.

This notebook removes that limitation:
1. **Retrain the 448 px selection run** (sections 1–6): the ViT-B/16 at 448 px with the learning rate [`vit_448px.ipynb`](vit_448px.ipynb) chose on validation (read from its results file), trained on the same 4,795 photos with the same code and settings. If that run's checkpoint is still in Drive, nothing is retrained.
2. **Save its validation probabilities** (section 7) for the 1,199 validation photos, from the last epoch, as `vit.ipynb` did for the 224 px ViT.
3. **Choose the attribute partner and `w` on validation** (sections 8–10), with the same rule as `consolidation.ipynb`: every attribute model × `w ∈ {0.5, …, 1.0}`, top-5 candidates from the ViT, best validation macro-F1 wins (on a tie, the larger `w`).
4. **Test once** (section 11) with the existing final 448 px ViT (`results/vit/vit_b16_448px_final_test_probs.npz`, trained on all 5,994 training photos) and the attribute models retrained on all training photos.

### Attribute models
The settings in the report (Table 3 / Appendix C), all on encoding A:

| Attribute model | Settings |
|---|---|
| Logistic regression | newton-cg, `C=1`, L2, `max_iter=500`, `tol=1e-4` |
| Random forest | 750 trees, Gini, `max_depth=30`, `max_features=0.1`, `min_samples_leaf=2` |
| k-NN | as chosen in `knn.ipynb` (`results/knn_best_params.json`) |
| k-NN + LDA | certainty-weighted attributes, as chosen in `knn_improvements.ipynb` (`results/knn_improvements.csv`) |
| Decision tree | default settings (Gini, unpruned) |

### How to run (Colab)
- Runtime → Change runtime type → **A100 GPU** (an L4 works, roughly 2× slower). The first time only, `kaggle.json` must be in Google Drive at `MyDrive/f20dl_resnet50/kaggle.json` (the same folder as the other image notebooks).
- Runtime → Run all. Expect about **15–20 minutes** of training on an A100 (one run of 25 epochs) plus a few minutes for the attribute models.
- If Colab disconnects, Run all again: training resumes from its last saved epoch.
- Results are written to `MyDrive/f20dl_resnet50/consolidation_448px/` (section 12 lists the files and where to put them in the repo).

*Code adapted from `vit_448px.ipynb` and `consolidation.ipynb` with help from Claude (Anthropic).*

## 1. Settings

Identical to `vit_448px.ipynb`, so the run name and settings match its selection runs and an existing checkpoint in Drive is reused.

In [ ]:
SMOKE_TEST = False     # True = tiny subset, 1 epoch: only checks that the code runs end to end

REPO_URL = "https://github.com/Flamingsides/f20dl-coursework-1.git"
BRANCH = "main"              # the branch Colab clones for scripts/ and the pipeline notebooks
DRIVE_FOLDER = "f20dl_resnet50"   # folder in MyDrive for the data cache, checkpoints and results (same as the CNN notebooks)

WEIGHTS = "IMAGENET1K_V1"    # torchvision's ViT-B/16 trained on ImageNet-1k at 224 px, with ImageNet normalisation

# Input size: 224 or 448 px. At 448 px each photo becomes 28 x 28 = 784 patches instead of 14 x 14 = 196, so the
# model sees finer detail; the pretrained position embeddings are resized to the larger grid (section 6).
# The photos are stored at 256 px (224 input) or 512 px (448 input) by scripts/prepare_images.py.
IMG_SIZE = 448
assert IMG_SIZE in (224, 448), "05_image_preparation.ipynb and the Drive caches are set up for 224 or 448 px"
STORE_PX = round(IMG_SIZE * 256 / 224)

BATCH_SIZE = 64              # photos per optimiser step, as in the 224 px run, so the learning rates compare directly
# Photos per forward/backward pass. 448 px has 4x the patches, so each batch of 64 is processed as 2 x 32 and the
# gradients are added up before the step (same result as one batch of 64: a ViT has no batch norm).
MICRO_BATCH = 32 if IMG_SIZE > 224 else BATCH_SIZE
EPOCHS = 25                  # same as the ResNet-50 runs
# Backbone learning rates to try. The 224 px run tried [0.0005, 0.001, 0.003] and picked 0.003, the top of its
# grid (validation macro-F1 0.70 at 0.001 -> 0.79 at 0.003), so this grid starts from 0.003 and goes up. For
# reference, the ViT paper fine-tunes with SGD at batch 512 over {0.001, 0.003, 0.01, 0.03}, i.e. up to about
# 0.004 at batch 64; gradient clipping (below) is what keeps the higher rates stable.
LR_GRID = [0.003, 0.006, 0.01]
WARMUP_EPOCHS, WEIGHT_DECAY = 1, 1e-4   # as in the ResNet-50 runs (the ViT paper used no weight decay)
HEAD_LR_MULT = 10            # the new 200-way head starts from random, so it learns faster
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0              # clip the gradient norm at 1, as in the ViT paper's fine-tuning recipe

if SMOKE_TEST:
    EPOCHS, WARMUP_EPOCHS, BATCH_SIZE, LR_GRID = 1, 0, 8, LR_GRID[:2]
    MICRO_BATCH = min(MICRO_BATCH, 4)

BASE_NAME = f"vit_b16_{IMG_SIZE}px" + ("_smoke" if SMOKE_TEST else "")
CONFIG = dict(pipeline="05_image_preparation", model="vit_b_16", weights=WEIGHTS, img_size=IMG_SIZE, store_px=STORE_PX,
              batch_size=BATCH_SIZE, micro_batch=MICRO_BATCH, epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS,
              head_lr_mult=HEAD_LR_MULT, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING,
              grad_clip=GRAD_CLIP, smoke_test=SMOKE_TEST)
print(f"{BASE_NAME}: backbone learning rates {LR_GRID} | batch {BATCH_SIZE} as {BATCH_SIZE // MICRO_BATCH} x {MICRO_BATCH}")
CONFIG

## 2. Environment

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from IPython.display import Image as ShowImage, display

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE.mkdir(parents=True, exist_ok=True)
    ROOT = Path("/content/f20dl-coursework-1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    OUT = DRIVE                                    # survives disconnects
    CKPT_DIR = DRIVE / "checkpoints"
else:
    DRIVE = None
    ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
    OUT = ROOT / "results" / "vit"
    CKPT_DIR = ROOT / "data" / "processed" / "checkpoints"   # git-ignored: checkpoints are ~0.35-1 GB each

FIGURES = OUT / "figures"
for d in (OUT, CKPT_DIR, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts"))

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True          # picks the fastest convolution algorithms for fixed-size inputs
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"                    # mixed precision: ~2x faster, same accuracy
GPU = torch.cuda.get_device_name() if DEVICE.type == "cuda" else ""

print(f"Colab: {IN_COLAB} | device: {DEVICE} {GPU} | {os.cpu_count()} CPU cores")
print(f"torch {torch.__version__} | outputs -> {OUT}")
if IN_COLAB and DEVICE.type != "cuda":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 or L4 GPU, then run all again.")

## 3. Photos and attribute tables

The cropped 448 px photos (512 px stored), restored from the Drive cache if it exists, as in `vit_448px.ipynb`. The attribute tables (`data/cub_features_A.csv`) are not in Git, so `dataPrep.ipynb` is run once if they are missing.

In [ ]:
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
SUFFIX = "" if STORE_PX == 256 else f"_{STORE_PX}"   # file names written by prepare_images.py --short-side
CSV = PROCESSED / f"cub_images_cropped{SUFFIX}.csv"   # the file 05_image_preparation.ipynb checks
IMG_DIR = PROCESSED / f"images_cropped{SUFFIX}"
CACHE_ZIP = DRIVE / f"cub_processed_{STORE_PX}px.zip" if IN_COLAB else None   # same cache names as the ResNet notebooks

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), check=True, cwd=ROOT)

# 1. Original photos (imagePrep needs them, and so does the before / after figure in 05)
if not (any((DATA / "raw").rglob("image_class_labels.txt")) or (ROOT / "CUB_200_2011").exists()):
    if IN_COLAB:
        token = DRIVE / "kaggle.json"
        if not token.exists():
            raise FileNotFoundError(f"Put your kaggle.json in Google Drive at {token} (see the top of this notebook)")
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(exist_ok=True)
        shutil.copy(token, kaggle_dir / "kaggle.json")
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
    run([sys.executable, "scripts/download_data.py"])

# 2. Cropped photos
if not CSV.exists() and IN_COLAB and CACHE_ZIP.exists():
    print("Restoring the cropped photos from the Drive cache...", flush=True)
    shutil.unpack_archive(CACHE_ZIP, DATA)
if not CSV.exists():
    if not (DATA / "cub_images.csv").exists():
        run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", PROCESSED / "_executed",
             "notebooks/imagePrep.ipynb"])
    run([sys.executable, "scripts/prepare_images.py", "--short-side", STORE_PX])
    if IN_COLAB:
        print("Caching the cropped photos in Drive...", flush=True)
        local_zip = Path("/content") / CACHE_ZIP.name     # build locally, then one copy to Drive (much faster)
        with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:   # JPEGs are already compressed
            z.write(CSV, CSV.relative_to(DATA))
            for p in IMG_DIR.rglob("*"):
                if p.is_file():
                    z.write(p, p.relative_to(DATA))
        shutil.copy(local_zip, CACHE_ZIP)

# 3. Attribute tables for the consolidation (generated by dataPrep.ipynb, not committed)
if not (DATA / "cub_features_A.csv").exists():
    run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", PROCESSED / "_executed",
         "notebooks/dataPrep.ipynb"])

print(f"{CSV.relative_to(ROOT)}: {len(pd.read_csv(CSV)):,} photos | cub_features_A.csv: {(DATA / 'cub_features_A.csv').exists()}")

## 4. Splits, transforms and loaders

The same `make_splits` (stratified 80/20 split of the official training photos, seed 42) as every image notebook, so the 1,199 validation photos are exactly the ones `consolidation.ipynb` used.

In [ ]:
from augmentation import BirdImages, make_splits, make_transforms, store_size, N_CLASSES, RANDOM_STATE

assert store_size(IMG_SIZE) == STORE_PX
train_transform, eval_transform = make_transforms(IMG_SIZE)   # the same augmentation choices at every size
train_df, val_df, test_df = make_splits(CSV)
if SMOKE_TEST:   # 10 species, a few photos each
    pick = lambda df, n: df[df.class_id <= 10].groupby("class_id").head(n)
    train_df, val_df, test_df = pick(train_df, 4), pick(val_df, 2), pick(test_df, 3)
full_train_df = pd.concat([train_df, val_df])     # all official training photos, for the final retrain
assert not set(full_train_df.image_id) & set(test_df.image_id)
assert not set(train_df.image_id) & set(val_df.image_id)

NUM_WORKERS = 0 if SMOKE_TEST else min(12, os.cpu_count() or 2)

def make_loader(df, train):
    return DataLoader(BirdImages(df, PROCESSED, train_transform if train else eval_transform),
                      batch_size=BATCH_SIZE, shuffle=train, drop_last=train, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0,
                      generator=torch.Generator().manual_seed(RANDOM_STATE) if train else None)

species = (pd.read_csv(CSV)[["class_id", "species"]].drop_duplicates()
           .set_index("class_id").sort_index().species.tolist())       # index = class_id - 1
print(f"train {len(train_df):,} | val {len(val_df):,} | all training {len(full_train_df):,} | "
      f"test {len(test_df):,} photos | input {IMG_SIZE} px | {NUM_WORKERS} loader workers")

## 5. Model and training code

Copied unchanged from `vit_448px.ipynb` (section 6 there explains each part).

In [ ]:
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision.models.vision_transformer import interpolate_embeddings
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def new_model():
    torch.manual_seed(RANDOM_STATE)
    model = vit_b_16(image_size=IMG_SIZE)
    if WEIGHTS:
        state = ViT_B_16_Weights[WEIGHTS].get_state_dict(progress=False)
        if IMG_SIZE != 224:   # 14x14 grid of pretrained position embeddings -> resized (bicubic) to the larger grid
            state = interpolate_embeddings(IMG_SIZE, model.patch_size, state)
        model.load_state_dict(state)
    model.heads = nn.Sequential(nn.Linear(model.hidden_dim, N_CLASSES))   # new 200-way head (random weights)
    return model.to(DEVICE, memory_format=torch.channels_last)


_m = vit_b_16(weights=None, image_size=IMG_SIZE)
_m.heads = nn.Sequential(nn.Linear(_m.hidden_dim, N_CLASSES))
print(f"ViT-B/16 at {IMG_SIZE} px: {len(_m.encoder.layers)} layers, {_m.hidden_dim} wide, patches "
      f"{_m.patch_size}x{_m.patch_size} px -> {(IMG_SIZE // _m.patch_size) ** 2} patches + [CLS], "
      f"{sum(p.numel() for p in _m.parameters()):,} weights")
del _m
if WEIGHTS:
    print("ViT weights' own preprocessing (224 px):", ViT_B_16_Weights[WEIGHTS].transforms())
print(f"Team eval_transform ({IMG_SIZE} px):        ", eval_transform)

def new_optimiser(model, lr_backbone, steps_per_epoch):
    head = list(model.heads.parameters())
    head_ids = {id(p) for p in head}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.SGD([{"params": backbone, "lr": lr_backbone},
                                 {"params": head, "lr": lr_backbone * HEAD_LR_MULT}],
                                momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)
    steps, warmup = EPOCHS * steps_per_epoch, WARMUP_EPOCHS * steps_per_epoch
    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        for xs, ys in zip(x.split(MICRO_BATCH), y.split(MICRO_BATCH)):   # one piece unless MICRO_BATCH < BATCH_SIZE
            with autocast():
                logits = model(xs)
                loss = criterion(logits, ys)
            scaler.scale(loss * len(ys) / len(y)).backward()   # gradients add up to those of the whole batch
            loss_sum += loss.item() * len(ys)
            correct += (logits.argmax(1) == ys).sum().item()
            seen += len(ys)
        scaler.unscale_(optimizer)                         # real gradients, so the clip applies to them
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
    return loss_sum / seen, correct / seen

@torch.no_grad()
def predict(model, loader):
    """Class probabilities, true labels (0-199) and mean cross-entropy (without smoothing)."""
    model.eval()
    probs, labels, loss_sum = [], [], 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
        loss_sum += nn.functional.cross_entropy(logits.float(), y, reduction="sum").item()
        probs.append(logits.float().softmax(1).cpu())
        labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return probs, labels, loss_sum / len(labels)

def scores(probs, labels):
    preds = probs.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "top5_accuracy": top_k_accuracy_score(labels, probs, k=5, labels=np.arange(N_CLASSES))}

def save_atomic(obj, path):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def fit(run_name, train_rows, val_rows, lr_backbone):
    """Train one run (resumable). With validation rows, keep the epoch with the best validation macro-F1;
    without, keep the last epoch. Returns the epoch history and a summary; the kept weights are in <run>_best.pt."""
    config = {**CONFIG, "run": run_name, "lr_backbone": lr_backbone, "train_photos": len(train_rows)}
    last, best_path = CKPT_DIR / f"{run_name}_last.pt", CKPT_DIR / f"{run_name}_best.pt"
    train_loader = make_loader(train_rows, train=True)
    val_loader = make_loader(val_rows, train=False) if val_rows is not None else None
    model = new_model()
    optimizer, scheduler = new_optimiser(model, lr_backbone, len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start, history, best = 0, [], {"epoch": 0, "val": None}
    if last.exists():
        ckpt = torch.load(last, map_location=DEVICE, weights_only=False)
        if ckpt["config"] != config:
            raise ValueError(f"{last.name} was saved with different settings - delete it to start this run again")
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        start, history, best = ckpt["epoch"] + 1, ckpt["history"], ckpt["best"]
        print(f"{run_name}: resuming after epoch {start}" if start < EPOCHS else f"{run_name}: already finished")

    for epoch in range(start, EPOCHS):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        row = {"epoch": epoch + 1, "train_loss": train_loss, "train_accuracy": train_acc}
        if val_loader is not None:
            val_probs, val_labels, val_loss = predict(model, val_loader)
            val = scores(val_probs, val_labels)
            row.update(val_loss=val_loss, **{f"val_{k}": v for k, v in val.items()})
        row["seconds"] = time.time() - t0
        history.append(row)
        keep = val_loader is None or best["val"] is None or val["macro_f1"] > best["val"]["macro_f1"]
        if keep:
            best = {"epoch": epoch + 1, "val": val if val_loader is not None else None}
            save_atomic({"model": model.state_dict(), "config": config, **best}, best_path)
        save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                     "history": history, "best": best, "config": config}, last)
        msg = f"{run_name} | epoch {epoch + 1:>2}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f}"
        if val_loader is not None:
            msg += (f" | val loss {val_loss:.3f} acc {val['accuracy']:.3f} macro-F1 {val['macro_f1']:.3f}"
                    f" top-5 {val['top5_accuracy']:.3f}")
        print(msg + f" | {row['seconds']:.0f}s" + ("  *" if keep and val_loader is not None else ""), flush=True)

    history = pd.DataFrame(history)
    history.to_csv(OUT / f"{run_name}_history.csv", index=False)
    del model, optimizer, scheduler
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return history, {"run": run_name, "lr_backbone": lr_backbone, "best_epoch": best["epoch"], "val": best["val"],
                     "train_minutes": round(history.seconds.sum() / 60, 1)}

## 6. Retrain the selection run with the chosen learning rate

The learning rate is the one `vit_448px.ipynb` chose by validation macro-F1, read from its results file, so nothing is chosen here. The run trains on the 4,795-photo training part and scores the validation photos after every epoch, with the same name and settings as the original selection run: if `vit_b16_448px_lr0.01_last.pt` is still in the Drive `checkpoints/` folder it is reused, otherwise the run is trained again (about 15–20 minutes on an A100). A retrained run can differ slightly from the original one (GPU kernels are not bit-for-bit deterministic).

In [ ]:
prev = json.loads((ROOT / "results" / "vit" / "vit_b16_448px_final_results.json").read_text())
BEST_LR = 0.003 if SMOKE_TEST else prev["config"]["lr_backbone"]
print(f"learning rate chosen in vit_448px.ipynb: {prev['config']['lr_backbone']:g} "
      f"(its validation macro-F1 by rate: {[(r['backbone lr'], round(r['val macro-F1'], 4)) for r in prev['selection']]})")

SEL_NAME = f"{BASE_NAME}_lr{BEST_LR:g}"
sel_history, sel_summary = fit(SEL_NAME, train_df, val_df, BEST_LR)
print(f"{SEL_NAME}: best validation macro-F1 {sel_summary['val']['macro_f1']:.4f} at epoch {sel_summary['best_epoch']} "
      f"({sel_summary['train_minutes']} min)")

## 7. Save the validation probabilities

As for the 224 px ViT in `vit.ipynb`, the probabilities come from the **last epoch** of the selection run, the same rule as the final model (which keeps its last epoch). That model never saw the validation photos, so the probabilities are fair for choosing the partner and weight.

In [ ]:
ckpt = torch.load(CKPT_DIR / f"{SEL_NAME}_last.pt", map_location=DEVICE, weights_only=False)
assert ckpt["epoch"] == EPOCHS - 1, "the selection run has not finished all its epochs"
model = new_model()
model.load_state_dict(ckpt["model"])
val_probs, val_labels, _ = predict(model, make_loader(val_df, train=False))
del model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
assert len(val_probs) == len(val_df) and (val_labels == val_df.class_id.to_numpy() - 1).all()

VAL_PROBS_PATH = OUT / f"{BASE_NAME}_val_probs.npz"
np.savez_compressed(VAL_PROBS_PATH, image_id=val_df.image_id.to_numpy(), class_id=val_df.class_id.to_numpy(),
                    probs=val_probs.astype(np.float32), model=SEL_NAME, epoch=ckpt["epoch"])
print(f"validation: {SEL_NAME}, last epoch ({ckpt['epoch'] + 1}), {len(val_probs):,} photos | {scores(val_probs, val_labels)}")
print(f"saved {VAL_PROBS_PATH}")

## 8. Load both sides for the consolidation

From here on everything runs on the CPU. The validation probabilities come from section 7; the test probabilities are those of the final 448 px ViT already in the repository. The attribute rows are matched to exactly the same photos, in the same order, and the checks stop the notebook if anything does not line up.

In [ ]:
import ast
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

assert not SMOKE_TEST, "the consolidation needs the full validation set: set SMOKE_TEST = False"
IMAGE_NAME = "ViT-B/16 (448 px)"
TEST_PROBS_PATH = ROOT / "results" / "vit" / "vit_b16_448px_final_test_probs.npz"
CON_OUT = (DRIVE / "consolidation_448px") if IN_COLAB else (ROOT / "results" / "consolidation")
(CON_OUT / "figures").mkdir(parents=True, exist_ok=True)
CUB = next((p.parent for p in (DATA / "raw").rglob("image_class_labels.txt")), ROOT / "CUB_200_2011")
K = 5                                            # candidates proposed by the image model
WEIGHTS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]         # w = weight of the image model (1.0 = image model alone)
CLASSES = np.arange(1, 201)

img_val, img_test = np.load(VAL_PROBS_PATH), np.load(TEST_PROBS_PATH)
feat_A = pd.read_csv(DATA / "cub_features_A.csv", index_col="image_id")
ATTR = [c for c in feat_A.columns if "::" in c]
assert len(ATTR) == 312
y_all = feat_A["class_id"]

val_ids, test_ids = img_val["image_id"], img_test["image_id"]
y_val = img_val["class_id"]
y_test = img_test["class_id"] if "class_id" in img_test.files else img_test["labels"].astype(int) + 1
assert set(val_ids) <= set(feat_A.index[feat_A.is_train == 1]), "validation photos must come from the official training split"
assert set(test_ids) == set(feat_A.index[feat_A.is_train == 0]), "test photos must be the official test split"
assert (y_all.loc[val_ids].values == y_val).all() and (y_all.loc[test_ids].values == y_test).all()
train_part_ids = feat_A.index[(feat_A.is_train == 1) & ~feat_A.index.isin(val_ids)]   # the 4,795 photos the ViT trained on
all_train_ids = feat_A.index[feat_A.is_train == 1]
p_img_val, p_img_test = img_val["probs"], img_test["probs"]

# Certainty-weighted attributes for k-NN + LDA (as in knn_improvements.ipynb and consolidation.ipynb)
labels_file = CUB / "attributes" / "image_attribute_labels_clean.txt"
labels = pd.read_csv(labels_file, sep=r"\s+", header=None, usecols=[0, 1, 3], names=["image_id", "attr_id", "certainty"])
names = pd.read_csv(CUB / "attributes" / "attributes.txt", sep=" ", header=None,
                    names=["attr_id", "attr_name"]).set_index("attr_id").attr_name
certainty = labels.pivot(index="image_id", columns="attr_id", values="certainty").rename(columns=names)
confidence = certainty.loc[feat_A.index, ATTR].replace({1: 0.0, 2: 0.5, 3: 0.75, 4: 1.0})
X_cw = (2 * feat_A[ATTR] - 1) * confidence      # present -> +confidence, absent -> -confidence, not visible -> 0

print(f"image model: {img_val['model']} (validation), vit_b16_448px_final (test)")
print(f"photos: train part {len(train_part_ids):,} | validation {len(val_ids):,} | all training {len(all_train_ids):,} | "
      f"test {len(test_ids):,}")

## 9. Attribute models

Each model is trained twice, mirroring the image model: on the 4,795-photo training part (to predict the validation photos) and on all 5,994 training photos (to predict the test photos).

In [ ]:
def sk_proba(make, X, fit_ids, pred_ids):
    model = make().fit(X.loc[fit_ids], y_all.loc[fit_ids])
    assert (model.classes_ == CLASSES).all()            # column j = species j + 1, as in the image probabilities
    return model.predict_proba(X.loc[pred_ids])


KNN_PARAMS = json.loads((ROOT / "results" / "knn_best_params.json").read_text())["params"]
improvements = pd.read_csv(ROOT / "results" / "knn_improvements.csv", index_col=0)
LDA_PARAMS = ast.literal_eval(improvements.loc["Extraction: LDA on certainty-weighted", "settings"])
print("k-NN:", KNN_PARAMS)
print("k-NN + LDA:", LDA_PARAMS)

ATTR_MODELS = {
    "Logistic regression": lambda f, p: sk_proba(lambda: LogisticRegression(
        C=1, solver="newton-cg", max_iter=500, tol=1e-4, random_state=RANDOM_STATE), feat_A[ATTR], f, p),
    "Random forest": lambda f, p: sk_proba(lambda: RandomForestClassifier(
        n_estimators=750, criterion="gini", max_depth=30, max_features=0.1, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1), feat_A[ATTR], f, p),
    "k-NN": lambda f, p: sk_proba(lambda: KNeighborsClassifier(**KNN_PARAMS), feat_A[ATTR], f, p),
    "k-NN + LDA (certainty-weighted)": lambda f, p: sk_proba(
        lambda: Pipeline([("lda", LDA()), ("knn", KNeighborsClassifier())]).set_params(**LDA_PARAMS), X_cw, f, p),
    "Decision tree": lambda f, p: sk_proba(lambda: DecisionTreeClassifier(random_state=RANDOM_STATE), feat_A[ATTR], f, p),
}

P_VAL, P_TEST = {}, {}
for name, fit_predict in ATTR_MODELS.items():
    t0 = time.time()
    P_VAL[name] = fit_predict(train_part_ids, val_ids)
    P_TEST[name] = fit_predict(all_train_ids, test_ids)
    print(f"{name}: {time.time() - t0:.0f}s")


def con_scores(y, pred):
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro", labels=CLASSES)}


single = pd.DataFrame(
    [{"model": IMAGE_NAME, "data": "images",
      **{f"val {k}": v for k, v in con_scores(y_val, p_img_val.argmax(1) + 1).items()},
      **{f"test {k}": v for k, v in con_scores(y_test, p_img_test.argmax(1) + 1).items()}}] +
    [{"model": name, "data": "attributes",
      **{f"val {k}": v for k, v in con_scores(y_val, P_VAL[name].argmax(1) + 1).items()},
      **{f"test {k}": v for k, v in con_scores(y_test, P_TEST[name].argmax(1) + 1).items()}}
     for name in ATTR_MODELS])
single.round(4)

## 10. Choose the attribute model and weight on the validation photos

`consolidate()` takes the ViT's top 5 species for each photo, scores each as `w × ViT probability + (1 − w) × attribute probability`, and returns the highest-scoring candidate. Every attribute model is tried with every weight on the validation photos; the pair with the best validation macro-F1 is kept (on a tie, the larger `w`, i.e. trusting the image model more). The test photos are not used here.

In [ ]:
def consolidate(p_img, p_tab, w, k=K):
    top = np.argsort(-p_img, axis=1)[:, :k]                                   # image model's k best species
    mixed = w * np.take_along_axis(p_img, top, 1) + (1 - w) * np.take_along_axis(p_tab, top, 1)
    return top[np.arange(len(top)), mixed.argmax(1)] + 1                      # back to class ids 1-200


def in_top_k(p, y, k=K):
    return (np.argsort(-p, axis=1)[:, :k] + 1 == y[:, None]).any(1).mean()


print(f"ceiling (true species in the ViT's top {K}): validation {in_top_k(p_img_val, y_val):.3f}")

rows = []
for name in ATTR_MODELS:
    for w in WEIGHTS:
        s = con_scores(y_val, consolidate(p_img_val, P_VAL[name], w))
        rows.append({"attribute model": name, "w (image)": w, "val accuracy": s["accuracy"], "val macro-F1": s["macro_f1"]})
weight_table = pd.DataFrame(rows)
best_per_model = (weight_table.sort_values(["val macro-F1", "w (image)"], ascending=[False, False])
                              .groupby("attribute model", sort=False).head(1).reset_index(drop=True))
BEST_MODEL, W = best_per_model.iloc[0]["attribute model"], float(best_per_model.iloc[0]["w (image)"])
print(f"chosen: {IMAGE_NAME} + {BEST_MODEL}, weight {W:.1f} image / {1 - W:.1f} attributes "
      f"(validation macro-F1 {best_per_model.iloc[0]['val macro-F1']:.4f}; ViT alone "
      f"{weight_table[weight_table['w (image)'] == 1.0]['val macro-F1'].iloc[0]:.4f})")

fig, ax = plt.subplots(figsize=(8, 4))
for name in ATTR_MODELS:
    t = weight_table[weight_table["attribute model"] == name]
    ax.plot(t["w (image)"], t["val macro-F1"], "o-", label=name, lw=2.5 if name == BEST_MODEL else 1.2)
ax.set_xlabel("weight of the image model (w); 1.0 = image model alone")
ax.set_ylabel("validation macro-F1")
ax.set_title(f"Choosing the attribute model and weight on the validation photos (top-{K} from {IMAGE_NAME})")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(CON_OUT / "figures" / "consolidation_448px_weights.png", dpi=150, bbox_inches="tight")
plt.show()
best_per_model.round(4)

## 11. Final test (run once)

The chosen pair and weight are applied once to the 5,794 test photos, next to each model on its own. Every pair is then listed on the test set for transparency (`fixed` = photos the ensemble gets right and the ViT alone gets wrong; `broken` = the reverse); the choice above was made on validation, not from this table. The last table shows how often the two chosen models agree.

In [ ]:
pred_img = p_img_test.argmax(1) + 1
pred_tab = P_TEST[BEST_MODEL].argmax(1) + 1
pred_con = consolidate(p_img_test, P_TEST[BEST_MODEL], W)
print(f"ceiling (true species in the ViT's top {K}): test {in_top_k(p_img_test, y_test):.3f}")

table5 = pd.DataFrame([
    {"model": f"Classical: {BEST_MODEL}", "data": "attributes", **con_scores(y_test, pred_tab)},
    {"model": f"Neural network: {IMAGE_NAME}", "data": "images", **con_scores(y_test, pred_img)},
    {"model": f"Model ensemble (top-{K}, w = {W:.1f})", "data": "both", **con_scores(y_test, pred_con)}])
display(table5.round(4))

img_ok = pred_img == y_test
rows = []
for _, r in best_per_model.iterrows():
    pred = consolidate(p_img_test, P_TEST[r["attribute model"]], r["w (image)"])
    ok = pred == y_test
    rows.append({"attribute model": r["attribute model"], "w (image)": r["w (image)"], "val macro-F1": r["val macro-F1"],
                 **{f"test {k}": v for k, v in con_scores(y_test, pred).items()},
                 "fixed": int(np.sum(ok & ~img_ok)), "broken": int(np.sum(~ok & img_ok))})
all_pairs = pd.DataFrame(rows)
display(all_pairs.round(4))

tab_ok, con_ok = pred_tab == y_test, pred_con == y_test
outcome = np.select([img_ok & tab_ok, tab_ok & ~img_ok, img_ok & ~tab_ok, pred_img == pred_tab],
                    ["Both correct", "Only the classical model is correct", "Only the neural network is correct",
                     "Both wrong (same species)"], "Both wrong (different species)")
order = ["Both correct", "Only the classical model is correct", "Only the neural network is correct",
         "Both wrong (same species)", "Both wrong (different species)"]
table6 = (pd.DataFrame({"outcome": outcome, "ensemble correct": con_ok})
            .groupby("outcome").agg(photos=("ensemble correct", "size"), ensemble_correct=("ensemble correct", "sum"))
            .reindex(order).fillna(0).astype(int))
table6["share (%)"] = (100 * table6.photos / len(y_test)).round(1)
fixed, broken = int(np.sum(con_ok & ~img_ok)), int(np.sum(~con_ok & img_ok))
print(f"{BEST_MODEL} and {IMAGE_NAME} agree on {np.mean(pred_img == pred_tab):.1%} of test photos; the ensemble fixed "
      f"{fixed} ViT mistakes and broke {broken} correct ViT predictions (net {fixed - broken:+d})")
table6[["photos", "share (%)", "ensemble_correct"]]

## 12. Files written

All in `MyDrive/f20dl_resnet50/` on Colab (`results/` locally). To commit them, download them into the repository:

| File | Put it in |
|---|---|
| `vit_b16_448px_val_probs.npz` (section 7) | `results/vit/` |
| `vit_b16_448px_lr0.01_history.csv` (only if the run was retrained) | `results/vit/` |
| `consolidation_448px/consolidation_448px_results.json` | `results/consolidation/` |
| `consolidation_448px/consolidation_448px_weights.csv` | `results/consolidation/` |
| `consolidation_448px/figures/consolidation_448px_weights.png` | `results/consolidation/figures/` |

In [ ]:
weight_table.to_csv(CON_OUT / "consolidation_448px_weights.csv", index=False)
results = {"image_model": {"name": IMAGE_NAME, "validation": str(img_val["model"]), "test": "vit_b16_448px_final",
                           "learning_rate": BEST_LR},
           "attribute_models": list(ATTR_MODELS), "k_candidates": K, "weights_tried": WEIGHTS,
           "chosen": {"attribute_model": BEST_MODEL, "weight_image": W},
           "validation_photos": int(len(y_val)), "test_photos": int(len(y_test)),
           "ceiling_top5": {"validation": float(in_top_k(p_img_val, y_val)), "test": float(in_top_k(p_img_test, y_test))},
           "single_models": single.to_dict("records"),
           "table5": table5.to_dict("records"),
           "all_pairs_test": all_pairs.to_dict("records"),
           "table6": table6.reset_index().to_dict("records"),
           "agreement": float(np.mean(pred_img == pred_tab)),
           "fixed": fixed, "broken": broken}
(CON_OUT / "consolidation_448px_results.json").write_text(json.dumps(results, indent=2))
sorted(str(p) for p in [VAL_PROBS_PATH, *CON_OUT.rglob("consolidation_448px*")])

## 13. Free the Drive space (optional)

The selection run's checkpoints take about 1.3 GB. Once the files above are saved, set `DELETE_CHECKPOINTS = True` and run this cell to delete them (the other notebooks' checkpoints and the photo cache are kept).

In [ ]:
DELETE_CHECKPOINTS = False

mine = sorted(CKPT_DIR.glob(f"{SEL_NAME}_*.pt"))
print(f"{len(mine)} checkpoints, {sum(p.stat().st_size for p in mine) / 1e9:.1f} GB")
if DELETE_CHECKPOINTS:
    for p in mine:
        p.unlink()
    print("deleted")